# MACURA on an unstable quadrotor — four algorithms, 20,000 real steps, and a flight demo

**MACURA** (*Trust the Model Where It Trusts Itself*, Frauenknecht et al., **ICML 2024**,
[arXiv:2405.19014](https://arxiv.org/abs/2405.19014)) makes the imagined-rollout length in
model-based RL **adaptive to the model's own epistemic uncertainty**: a rollout continues while the
dynamics ensemble's members agree and truncates the instant they disagree. This notebook transfers
that mechanism to a domain the paper never tested — an **unstable quadrotor hover task** where model
errors compound within a handful of steps — and runs the paper's full four-way comparison
(**MACURA** vs **MBPO** vs **M2AC** vs model-free **SAC**) on a shared SAC backbone and a shared
probabilistic ensemble, for **20,000 real environment steps**. The hypothesis under test: MACURA ≳
M2AC ≳ MBPO in sample efficiency, all above SAC, with MACURA the most stable (lowest failure rate).
At the end we load each algorithm's best checkpoint and record a **12-second flight video per
algorithm**, saved to Google Drive, plus a side-by-side reel — the results are something you can
*watch*, not just plot.

### How to read this notebook

- **The `SMOKE` flag** (set in the config cell below) is the one switch you touch.
  `SMOKE = True` runs the entire notebook end-to-end in minutes (~1.5k steps, 1 seed) to validate
  the pipeline; `SMOKE = False` runs the real 20k-step matrix (4 algorithms × 3 seeds). Nothing else
  changes — same code path, same figures, same videos.
- **All hyperparameters live in `configs/macura_drone.yaml`** — the single source of truth. The
  notebook only selects a profile; it hard-codes nothing.
- **Everything lands in Google Drive** and survives a runtime reset:
  `{DRIVE}/logs` (training curves), `{DRIVE}/checkpoints` (best policy + ensemble per run),
  `{DRIVE}/plots` (every figure), `{DRIVE}/videos` (the demo clips). Smoke artifacts go to
  `{DRIVE}/smoke/...` so they never overwrite real results.
- **Training is idempotent**: a run whose log JSON already exists in Drive is skipped and reloaded.
  Compute — not samples — is the Colab bottleneck (README §7), so nothing is ever retrained by
  accident. Delete a log file to force a rerun.

### 0.1 Google Drive + GPU

Everything below assumes a CUDA GPU — the ensemble and four SAC learners are built on `cuda`, and
there is deliberately **no silent CPU fallback** (a CPU run would take days and produce nothing
comparable). If the assert fires: *Runtime → Change runtime type → GPU*, then rerun. We also set
`MUJOCO_GL=egl` up front so any MuJoCo rendering is headless.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, warnings
os.environ['MUJOCO_GL'] = 'egl'      # headless rendering (no display on Colab)
warnings.filterwarnings('ignore')
import torch
assert torch.cuda.is_available(), 'No GPU. Enable one: Runtime > Change runtime type > GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))

### 0.2 GitHub token

The repo is private, so the clone needs a GitHub personal-access token. It is read from Colab
Secrets (`GITHUB_TOKEN`) or a hidden prompt, kept only in this process's environment, and the setup
script scrubs it from the git remote after cloning — it never lands on disk or in git config.

In [ ]:
TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
os.environ['GITHUB_TOKEN'] = TOKEN
print('token loaded')

### 0.3 Clone + install

`bash/setup_colab.sh` does the one-time bring-up: clone this repo (or `git pull` if present), clone
`mujoco_menagerie` for the Skydio X2 model, `pip install` the requirements, build
`gym-pybullet-drones` from source with `--no-deps` (it is not on PyPI, and its pins would corrupt
Colab's numpy 2 stack), repair numpy consistency, and create the Drive folders. Watch its output for
the per-step timings and the final "DONE" banner; a numpy warning here means *Runtime → Restart
session*, then rerun from the top.

In [ ]:
%%bash
set -e
curl -sH "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/silvergjeka22/macura-drone/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

### 0.4 Imports, config, and the `SMOKE` switch

We import the project's pure-function modules and load `configs/macura_drone.yaml`. Two details
matter here. First, the **`fast` profile in the YAML *is* the 20k-step run** (`total_env_steps:
20000`) — the notebook deep-merges it over the base config rather than defining numbers of its own.
The merge must be **recursive** because the profile overrides nested keys (e.g.
`rollout.mbpo.fixed_gradient_steps`, part of the UTD-fairness fix in Part 3). Second, `SMOKE` picks
between the tiny validation config and the real one, and routes all outputs to `{DRIVE}/smoke/` vs
`{DRIVE}/` accordingly. Check the printed summary: it should show 20000 steps and 3 seeds when
`SMOKE = False`.

In [ ]:
import sys
if '/content/macura-drone' not in sys.path:
    sys.path.insert(0, '/content/macura-drone')
import copy, glob, hashlib, json, yaml
import numpy as np
import pandas as pd
import pybullet, gymnasium, stable_baselines3
from IPython.display import Video, display

from envs import drone_env
from models import ensemble as ens
from algorithms import sac as sac_mod
from algorithms import macura as macura_mod
from training import train as trainer
from viz import plots

PYB_VER = getattr(pybullet, '__version__', None) or pybullet.getAPIVersion()
print('torch', torch.__version__, '| sb3', stable_baselines3.__version__,
      '| pybullet', PYB_VER, '| gymnasium', gymnasium.__version__, '| numpy', np.__version__)

CFG = yaml.safe_load(open('/content/macura-drone/configs/macura_drone.yaml'))
DRIVE = CFG['experiment']['drive_root']

SMOKE = True    # True = whole notebook end-to-end in minutes | False = the real 20k-step matrix

def deep_merge(base, over):
    # recursive dict merge: `over` wins, nested dicts merge (the fast profile
    # overrides nested keys like rollout.mbpo.fixed_gradient_steps)
    out = copy.deepcopy(base)
    for k, v in over.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_merge(out[k], v)
        else:
            out[k] = copy.deepcopy(v)
    return out

def make_cfg(smoke=False):
    # every quantity comes from configs/macura_drone.yaml; the notebook only selects
    c = deep_merge(CFG, CFG.get('profiles', {}).get('fast', {}))   # fast profile = the 20k run
    if smoke:
        c = deep_merge(c, {
            'experiment': dict(total_env_steps=1500, warmup_random_steps=300,
                               eval_every_steps=500, eval_episodes=2, seeds=[0]),
            'rollout':    dict(num_rollouts=100, freq_steps=250),
            'selection':  dict(start_step=500, eval_every=500, final_eval_episodes=2),
        })
    c['env']['backend'] = 'pybullet'
    c['env']['mjcf_scene'] = '/content/mujoco_menagerie/skydio_x2/scene.xml'
    return c

cfg_full  = make_cfg(smoke=False)       # the 20k-step configuration
cfg_smoke = make_cfg(smoke=True)        # tiny end-to-end validation
cfg_run   = cfg_smoke if SMOKE else cfg_full

SMOKE_DIR = f'{DRIVE}/smoke'            # smoke artifacts never overwrite real results
RUN_DIR   = SMOKE_DIR if SMOKE else DRIVE
PLOTS     = f'{RUN_DIR}/plots'
for d in (f'{RUN_DIR}/logs', f'{RUN_DIR}/checkpoints', f'{RUN_DIR}/videos', PLOTS,
          f'{SMOKE_DIR}/logs', f'{SMOKE_DIR}/checkpoints'):
    os.makedirs(d, exist_ok=True)

print(f"SMOKE={SMOKE} | steps {cfg_run['experiment']['total_env_steps']}"
      f" | seeds {cfg_run['experiment']['seeds']}"
      f" | eval every {cfg_run['experiment']['eval_every_steps']}"
      f" | real_ratio {cfg_run['sac']['real_ratio']} | outputs -> {RUN_DIR}")

## Part 1 — What MACURA introduces

Model-based RL trains the policy mostly on *imagined* transitions from a learned dynamics model, so
it needs far fewer real interactions than model-free RL. The catch is **model exploitation** (paper
Sec. 1): the model's small one-step errors compound multiplicatively along a rollout, and a policy
trained on long imagined trajectories learns to exploit fantasy dynamics — returns collapse exactly
when the rollouts get long enough to be useful. MBPO's answer was a hand-tuned *schedule* for the
rollout length; MACURA's answer is to let **each individual rollout decide for itself** how far the
model can be trusted, using the model's own uncertainty. The figure below is a synthetic
illustration of the core tension: error grows geometrically with horizon, so there is a horizon
beyond which imagined data hurts — and that horizon moves during training, which is why a fixed
schedule struggles.

In [ ]:
plots.plot_model_error_growth(save_path=f'{PLOTS}/p1_model_error_growth.png');
# Takeaway: compounding model error makes LONG fixed horizons poisonous; the right
# truncation point depends on current model quality, so it should be adaptive.

### 1.2 The four algorithms in this study

All four share the **same SAC learner** and (for the model-based three) the **same probabilistic
ensemble**, identical evaluation seeds, and one exploration scheme — the *only* thing that differs
is the rollout strategy, so any performance gap is attributable to that strategy alone
(`project.md` §6).

| Algorithm | Type | Rollout strategy | Uncertainty use | Implementation |
|---|---|---|---|---|
| **MACURA** | model-based | **adaptive truncation**: continue while u_GJS < κ (Alg. 2) | GJS drives adaptive κ (Eq. 21) | `algorithms/macura.py` — `compute_gjs`, `update_kappa`, `macura_rollout`, `gradient_steps` |
| **MBPO** | model-based | **fixed** truncated-linear length schedule (Janner 2019) | none | `algorithms/mbpo.py` — `rollout_length`, `mbpo_rollout` |
| **M2AC** | model-based | fixed length, then **mask** the least-trustworthy transitions (Pan 2020) | filter + reward penalty | `algorithms/m2ac.py` — `m2ac_rollout` |
| **SAC** | model-free | no model rollouts | — | `algorithms/sac.py` — `build_sac`, `sac_update` |

The Dyna loop that wires them together (real steps → ensemble retrain → branched rollouts → mixed
SAC updates) is one shared function: `training/train.py::train_one`. What each pairing reveals:
**MACURA vs SAC** — does the model help at all? **MACURA vs MBPO** — does *adaptivity* matter, or is
a schedule fine? **MACURA vs M2AC** — is adapting the *length* better than *masking* bad samples?

### 1.3 The probabilistic ensemble — where uncertainty comes from

All three model-based algorithms learn the dynamics with an **ensemble of Gaussian MLPs**
(`models/ensemble.py`), each predicting a distribution over the next-state *delta*. Each member
trains on its own bootstrap resample of the real data, so in regions dense with data the members
converge to the same answer, while out-of-distribution they **disagree** — and that disagreement is
exactly *epistemic* uncertainty, the "I haven't seen this" signal MACURA needs (paper Sec. 2.2,
5.1). The aleatoric part (genuine noise) lives in each member's predicted variance instead. The cell
below trains our actual ensemble implementation on a 1-D toy problem: watch the members agree inside
the shaded training region and fan out beyond it.

In [ ]:
plots.plot_ensemble_toy_1d(cfg_run['ensemble'], device='cuda',
                           save_path=f'{PLOTS}/p1_ensemble_toy.png');
# Takeaway: member disagreement IS epistemic uncertainty — near the data the blue
# curves overlap; away from it they diverge, and that spread is measurable.

### 1.4 Measuring disagreement: the GJS uncertainty (Eq. 15–19)

MACURA needs a *number* for "how much do the members disagree at (s, a)". The plain Jensen–Shannon
divergence has no closed form for Gaussians, so the paper uses the **geometric Jensen–Shannon
divergence**: u_GJS(s, a) averages D_GJS over all member pairs (**Eq. 15–16**), where each pair term
is built from two KL divergences to the *geometric-mean* Gaussian (**Eq. 17**) with precision-
averaged covariance (**Eq. 18**) and mean (**Eq. 19**) — all closed-form for our diagonal Gaussians.
This is implemented, exactly, in `compute_gjs` (`algorithms/macura.py`); the "trust region" is then
the set of states with u_GJS < κ (**Eq. 20**).

One correctness note that this project had to fix (Part 3 details it): u_GJS **sums over state
dimensions**, so the ensemble must be trained on **normalized** delta targets — otherwise dimensions
with large physical scales (position) drown out small ones (angular rates) in both the training loss
and the variance bounds, and the uncertainty signal degenerates.

In [ ]:
plots.plot_gjs_vs_spread(save_path=f'{PLOTS}/p1_gjs_vs_spread.png');
# Takeaway: u_GJS (compute_gjs, Eq. 15) rises monotonically as member means spread
# apart — it is a well-behaved, closed-form dial from "members agree" to "members disagree".

### 1.5 The adaptive threshold κ and rollout truncation (Eq. 21, Algorithm 2)

How much disagreement is *too much*? A fixed threshold can't work — u_GJS falls by orders of
magnitude as the ensemble learns. MACURA makes the threshold self-tuning (**Eq. 21**): at each
rollout round, take the **ζ-quantile of the first-step uncertainties** of the M branched rollouts
(first-step = starting from *real* states, so it's an upper bound on "certain"), scale it by the
knob **ξ**, and average over rounds: κ = ξ/K · Σ û_GJS,k. That is `update_kappa`. The rollout itself
(**Algorithm 2**, `macura_rollout`) then branches from M real states and, at every imagined step,
keeps a branch alive only while u_GJS < κ and the analytic termination hasn't fired — up to T_max.
Early in training most branches die at step 1–2; late in training they run long. ξ (config
`rollout.macura.xi`) is the *one* tunable knob the paper emphasizes.

In [ ]:
plots.plot_rollout_truncation_cartoon(save_path=f'{PLOTS}/p1_truncation_cartoon.png');
# Takeaway: a rollout is truncated the moment its uncertainty crosses kappa —
# short leash when the model is shaky, long leash when it has earned trust.

### 1.6 Update-to-data scaling (Eq. 22) — and a config sanity check

More imagined data supports more gradient updates. MACURA scales the number of SAC updates per real
step with model-buffer fullness (**Eq. 22**): G = ⌈G_max · |D_mod| / |D_mod|_max⌉, implemented in
`gradient_steps`. Early on (near-empty model buffer) that is ~1 update/step; at a full buffer it is
G_max. MBPO/M2AC use a fixed G — Part 4 runs a control experiment to make sure MACURA's win is not
just this extra-updates effect. The cell below demos Eq. 22 numerically, then prints the **fully
resolved 20k-step configuration** — read it once now; every number you see later comes from here.

In [ ]:
for frac in (0.05, 0.25, 0.50, 1.00):
    g = macura_mod.gradient_steps(int(frac * 1000), 1000,
                                  cfg_full['sac']['gradient_steps_max'], adaptive=True)
    print(f'model buffer {int(frac*100):3d}% full -> G = {g} SAC updates per real step (Eq. 22)')

resolved = {k: v for k, v in cfg_full.items() if k != 'profiles'}
print('\n──── resolved 20k-step config (deep-merged fast profile; single source of truth) ────')
print(yaml.dump(resolved, sort_keys=False))

## Part 2 — The environment: an unstable quadrotor

The task: a **Crazyflie 2.x quadrotor** (PyBullet, via `gym-pybullet-drones`) must take off and
**hover at (0, 0, 1 m)**. Observation = 13-dim kinematic state `[pos_error(3), quaternion(4),
lin_vel(3), ang_vel(3)]`; action = 4 per-rotor commands in [-1, 1]; reward = dense quadratic cost on
pose/velocity/tilt/effort plus an alive bonus; an episode **terminates as a failure** when the drone
tilts past ~57°, leaves a 1.5 m envelope, or hits the floor (`envs/drone_env.py`). Everything in
this part builds one intuition: unlike the paper's MuJoCo locomotion tasks, a quadrotor **balances
on its own thrust** — it is open-loop unstable, so a small dynamics-model error grows within a few
steps into a predicted state the real drone would never reach. That is precisely the regime where a
fixed rollout horizon over-imagines and adaptive truncation should matter most (`project.md` §2).
The env is deliberately "low-step friendly": near-hover initialization, action repeat, and a smooth
model-predictable reward, so 20k real steps are enough for clear curves.

In [ ]:
env, obs_dim, act_dim = drone_env.make_env(cfg_run['env'], seed=0, render=True)
reward_fn = drone_env.known_reward_fn(cfg_run['env'])   # analytic reward, shared by all rollouts
done_fn   = drone_env.termination_fn(cfg_run['env'])    # analytic failure envelope, ditto
obs, _ = env.reset(seed=0)
print('obs_dim', obs_dim, '| act_dim', act_dim,
      '| HOVER_RPM', round(env.HOVER_RPM, 1), '| MAX_RPM', round(env.MAX_RPM, 1))
print('initial pos_error (m):', np.round(obs[0:3], 3),
      '| target:', cfg_run['env']['target_position'])

### 2.2 What it looks like

A filmstrip of the drone under **random actions** — frames a few steps apart. Watch the attitude
degrade frame to frame: random thrust commands tip it over almost immediately. Every training run
starts from policies like this.

In [ ]:
plots.render_filmstrip(env, policy='random', n_frames=6, steps_between=8, seed=0,
                       title='Random policy — the starting point of every run',
                       save_path=f'{PLOTS}/p2_filmstrip.png');

### 2.3 The "before" video

A ~12-second clip of the untrained (random) behaviour, resetting on every crash. Keep this clip in
mind — Part 5b records the same 12 seconds from each algorithm's *best* policy, and the contrast is
the demo.

In [ ]:
vid = f'{RUN_DIR}/videos/env_demo_random.mp4'
plots.record_env_video(env, policy='random', seconds=12, fps=30, save_path=vid)
print('saved', vid)
display(Video(vid, embed=True, width=520))

### 2.4 Instability, quantified

Per-step traces for one episode of random actions and one of **constant mid-thrust** (zero action =
hover RPM on all rotors, i.e. no feedback at all). Both blow up — the constant-thrust one drifts and
tips within a couple of hundred steps, which is the textbook signature of an open-loop unstable
plant: even the *right average input* fails without feedback. A dynamics model's small errors get
amplified by exactly this mechanism.

In [ ]:
rec_rand = plots.collect_rollout(env, policy='random', num_steps=200, seed=1)
plots.plot_rollout_traces(rec_rand, title='Random actions: tilt/altitude diverge within tens of steps',
                          save_path=f'{PLOTS}/p2_traces_random.png')
rec_hover = plots.collect_rollout(env, policy='hover', num_steps=200, seed=1)
plots.plot_rollout_traces(rec_hover, title='Constant hover thrust (no feedback): still drifts off and dies',
                          save_path=f'{PLOTS}/p2_traces_hover.png');

### 2.5 The data regime the ensemble must learn

Histograms of the states visited under random actions — this is roughly what the ensemble sees
during warmup. Note the wildly different physical scales across quantities (metres vs radians vs
rad/s): this is why the ensemble's **delta targets must be normalized** before the GJS sum over
dimensions means anything (the Part-3 fix).

In [ ]:
plots.plot_state_distributions(env, num_episodes=15, num_steps=100,
                               save_path=f'{PLOTS}/p2_state_dists.png');

### 2.6 The reward is model-friendly by design

MBRL needs a reward the model can predict, so the reward is a smooth quadratic bowl peaked at the
hover target (1-D slice below, then the 2-D (x, y) heatmap). This is also why the reward is *known
analytically* to the rollouts (`known_reward_fn`) — more on that fairness choice in Part 3. Dense
reward is a firm requirement: MBRL is the wrong tool for sparse rewards, and this project does not
pretend otherwise (`project.md` §8).

In [ ]:
plots.plot_reward_landscape(cfg_run, save_path=f'{PLOTS}/p2_reward_1d.png')
plots.plot_reward_heatmap_2d(cfg_run, save_path=f'{PLOTS}/p2_reward_2d.png');

### 2.7 Actions do what they should

Last sanity check before training: sweep a constant total-thrust command and record the altitude
change. The response should be monotone — more thrust, more climb. If this plot were flat or
non-monotone, no algorithm downstream could learn. We close the study env afterwards.

In [ ]:
plots.plot_action_response(env, save_path=f'{PLOTS}/p2_action_response.png')
env.close()

## Part 3 — Paper vs project: the fairness story

A comparison is only worth running if no algorithm gets a hidden edge. This project's audit
(`tasks.md` Phase 4) found six correctness/fairness issues that could hand MACURA an accidental win
— or cripple it. All six are now fixed **in the library modules** (not patched in the notebook), and
each is verified by a cell in this notebook:

1. **Pink noise is real** — `exploration.type: pink_noise` now builds a genuine per-episode 1/f
   generator (`_PinkNoise` in `training/train.py`, FFT method, reset every episode), used by
   `_explore` for **all four** algorithms. Previously white noise was silently substituted.
   *Verified in 3.1 below.*
2. **GJS is scale-balanced** — `models/ensemble.py` now normalizes the **delta targets** (not just
   inputs). The NLL then weights every state dimension equally and the log-variance bounds act on a
   per-dimension scale, so `compute_gjs`'s sum over dimensions is meaningful instead of being
   dominated by the largest-magnitude dims. Without this, κ and adaptive truncation ride on noise.
   *Verified by κ/GJS behaviour in 4.1 and figures 5.6–5.7.*
3. **SAC baseline is standard** — the model-free baseline now runs `sac.baseline_gradient_steps: 1`
   update per real step (standard SAC), not the model-based ceiling of 8–20, which destabilizes
   plain SAC and would have been a strawman. *Asserted in 6.2.*
4. **MBPO's schedule is tuned for the budget** — `rollout_schedule: [1, 10, 1000, 12000]` actually
   ramps 1→10 inside the 20k run (the old `[1, 15, 1, 15000]` never left length ~1). MBPO is now a
   real fixed-horizon competitor that then over-imagines on the unstable drone — the *honest* way
   MACURA can beat it. *Asserted in 6.2.*
5. **UTD confound controlled** — MACURA is run both with adaptive G (Eq. 22) and with G fixed at
   G_max, so the win can't be attributed to extra gradient steps. Fixed UTD for MBPO/M2AC is also
   pinned equal to G_max in the fast profile. *Figure 5.9.*
6. **SB3 manual-training API verified** — the installed Stable-Baselines3's logger setup and
   replay-buffer layout are checked against what `algorithms/sac.py` / `_bulk_add` assume, before
   any training. *Cell 4.1.*

### 3.1 Exploration noise

**Paper does:** pink (1/f, temporally correlated) exploration noise for MACURA (Sec. 6.1/7,
following Eberhard et al. 2023), while baselines keep their native exploration.
**This project does:** pink noise for **all four** algorithms — deterministic policy mean plus a
per-episode 1/f sequence, identical scheme and scale.
**Why:** temporally correlated actions produce coherent flight excursions instead of thrust jitter,
which is better data for a dynamics model; giving it only to MACURA would confound "adaptive
rollouts help" with "better exploration helps".
**Caveat:** this departs from the paper's exact setup — deliberately, in fairness's favour.
The cell first *proves* the trainer's actual noise generator is temporally correlated (the old bug
was precisely that it wasn't), then shows the illustrative time-series/PSD comparison.

In [ ]:
horizon = cfg_run['env']['max_episode_steps']
pink  = trainer._make_noise({'type': 'pink_noise',  'scale': 0.1}, act_dim, horizon, seed=0)
white = trainer._make_noise({'type': 'white_noise', 'scale': 0.1}, act_dim, horizon, seed=0)
ps = np.array([pink.sample()  for _ in range(horizon)])
ws = np.array([white.sample() for _ in range(horizon)])
lag1 = lambda x: float(np.corrcoef(x[:-1, 0], x[1:, 0])[0, 1])
print(f'lag-1 autocorrelation of the TRAINING noise: pink {lag1(ps):+.2f} | white {lag1(ws):+.2f}')
assert isinstance(pink, trainer._PinkNoise) and lag1(ps) > 0.3, 'pink noise not actually correlated!'
assert cfg_run['exploration']['type'] == 'pink_noise', 'config must give ONE scheme to all four'
print('fix #1 verified: configured pink noise is genuinely 1/f-correlated, reset per episode')
plots.plot_noise_comparison(save_path=f'{PLOTS}/p3_noise.png');

### 3.2 Rollout strategy per algorithm

**Paper does:** compares adaptive truncation (MACURA) against MBPO's scheduled length and M2AC's
masking on MuJoCo locomotion. **This project does:** the same three strategies plus model-free SAC,
implemented over one shared ensemble and learner. **Why:** isolating the rollout strategy is the
entire point of the study. **Caveat:** our M2AC uses the *same GJS* uncertainty as MACURA for its
masking (the paper's M2AC uses a brittler one-vs-rest estimate), which if anything *strengthens*
M2AC — a conservative choice (`tasks.md` Phase 4, "faithful M2AC"). The schematic shows what each
strategy's rollout length looks like over training.

In [ ]:
plots.plot_rollout_strategy_schematic(save_path=f'{PLOTS}/p3_strategies.png');

### 3.3 Known reward and termination

**Paper does:** assumes a known reward — Algorithm 2 computes r(s_t, a_t) analytically during
imagined rollouts. **This project does:** the same, and also applies the analytic **failure
envelope** (`termination_fn`) inside every model rollout, for all three model-based algorithms.
**Why:** a learned reward head would add a second error source that differs across runs; and without
the shared termination check the ensemble happily hallucinates flight *past a crash*, poisoning the
buffer. **Caveat:** on a real robot the reward is designed (so "known" is realistic), but the
termination envelope is a modelling choice.

### 3.4 SB3 backbone instead of mbrl-lib

**Paper does:** builds on mbrl-lib. **This project does:** Stable-Baselines3 SAC as the learner,
driven manually by our own Dyna loop, plus a self-contained PyTorch ensemble. **Why:** mbrl-lib is
unmaintained and not installable on Colab's Python 3.12 / numpy 2 stack. **Caveat:** driving SB3
without `.learn()` relies on version-sensitive internals (`_setup_learn` logger, replay-buffer
layout) — exactly why cell 4.1 verifies them against the installed version before training.

### 3.5 Domain: locomotion → unstable quadrotor

**Paper does:** MuJoCo locomotion (Hopper, Walker2d, Humanoid…) — smooth, mostly self-stabilizing.
**This project does:** an open-loop-unstable quadrotor hover, in 20k real steps. **Why:** it is the
sharpest test of the core claim — where model errors compound fastest, adaptivity should matter
most. **Caveat:** a transfer, not a reproduction of the paper's numbers; a negative result would
still be a finding, and Part 6 reports whatever the data says.

**The hypothesis this notebook tests** (fixed before looking at results): *at 20k real steps,*
**MACURA ≳ M2AC ≳ MBPO in sample efficiency, all above SAC; MACURA has the lowest failure rate and
the highest final return** — and MACURA's advantage survives fixing G (the UTD control).

## Part 4 — Running the comparison

### 4.1 Smoke test (always runs, minutes)

Before spending GPU-hours we validate the full pipeline on a ~1.5k-step budget: the SB3
manual-training API (fix #6), then one tiny run of **each** algorithm, then hard checks that κ is
finite and updating, rollout lengths are logged and bounded, model rollouts respect the termination
envelope, and best checkpoints reload from Drive. Smoke artifacts live under `{DRIVE}/smoke/` and
are themselves idempotent. First, the SB3 checks — these would crash *here*, loudly, rather than
1500 steps into a training run.

In [ ]:
import inspect
probe = sac_mod.build_sac(obs_dim, act_dim, {**cfg_smoke['sac'], 'buffer_size': 1000}, 'cuda', 0)

# (a) manual .train() needs a logger — _setup_learn must have set one (tasks.md Phase 2)
assert getattr(probe, '_logger', None) is not None or getattr(probe, 'logger', None) is not None, \
    'SB3 logger unset after _setup_learn -> agent.train() would crash; fix algorithms/sac.py'

# (b) replay_buffer.add ordering + memory layout assumed by add_to_buffer/_bulk_add
sig = list(inspect.signature(probe.replay_buffer.add).parameters)
assert sig[:6] == ['obs', 'next_obs', 'action', 'reward', 'done', 'infos'], f'SB3 add() changed: {sig}'
assert probe.replay_buffer.optimize_memory_usage is False

# (c) the vectorized _bulk_add must write EXACTLY what SB3's own add() writes
rng = np.random.default_rng(0); n = 17
o  = rng.normal(size=(n, obs_dim)).astype(np.float32)
a  = rng.uniform(-1, 1, (n, act_dim)).astype(np.float32)
no = rng.normal(size=(n, obs_dim)).astype(np.float32)
r  = rng.normal(size=n).astype(np.float32)
d  = (rng.random(n) < 0.2).astype(np.float32)
ref = sac_mod.build_replay_buffer(probe, 1000)
for i in range(n):
    sac_mod.add_to_buffer(ref, o[i], a[i], no[i], float(r[i]), bool(d[i]))
trainer._bulk_add(probe, o, a, no, r, d)
for attr in ('observations', 'next_observations', 'actions', 'rewards', 'dones'):
    assert np.allclose(getattr(probe.replay_buffer, attr)[:n], getattr(ref, attr)[:n]), attr
del probe, ref
print('fix #6 verified on SB3', stable_baselines3.__version__,
      '— logger set, add() ordering OK, _bulk_add == per-row add')

Now the four tiny training runs. Each prints a live line per evaluation — return, failure rate, and
the realised real/imagined batch split (should hover near the fixed 5% `real_ratio` target for the
model-based three). Existing smoke logs are reused.

In [ ]:
smoke_runs = []
for algo in cfg_smoke['experiment']['algorithms']:
    lp = f'{SMOKE_DIR}/logs/{algo}_seed0.json'
    if os.path.exists(lp):
        print('reuse smoke log:', algo)
        smoke_runs.append(json.load(open(lp)))
    else:
        smoke_runs.append(trainer.train_one(algo, cfg_smoke, SMOKE_DIR, seed=0))
print('smoke training complete:', [r['algo'] for r in smoke_runs])

In [ ]:
from stable_baselines3 import SAC

# (a) kappa: finite, updated every rollout round; rollout lengths logged and bounded
mac = next(r for r in smoke_runs if r['algo'] == 'macura')
ks, kv = zip(*mac['kappa'])
assert len(kv) >= 2 and all(np.isfinite(kv)), 'kappa must be finite and update each round'
rl = [l for _, l in mac['rollout_length']]
t_max = cfg_smoke['rollout']['macura']['t_max']
assert all(0 <= l <= t_max for l in rl)
gj = [g for _, g in mac['gjs']]
assert all(np.isfinite(gj)) and min(gj) > 0, 'first-step GJS must be finite and positive'
print(f'kappa over {len(kv)} rounds: {kv[0]:.4f} -> {kv[-1]:.4f} | '
      f'mean rollout length {np.mean(rl):.2f} (t_max={t_max}) | first-step GJS {gj[0]:.4f} -> {gj[-1]:.4f}')

# (b) model rollouts must respect termination: under an always-done envelope no
#     rollout may contribute more than one step
agent = SAC.load(f'{SMOKE_DIR}/checkpoints/macura_seed0_best.zip', device='cuda')
world = ens.build_ensemble(cfg_smoke['ensemble'], obs_dim, act_dim, 'cuda')
ens.load_ensemble(world, f'{SMOKE_DIR}/checkpoints/macura_seed0_best_ensemble.pt')
tmp_env, _, _ = drone_env.make_env(cfg_smoke['env'], seed=7)
starts = np.stack([tmp_env.reset(seed=100 + i)[0] for i in range(64)])
tmp_env.close()
_, diag = macura_mod.macura_rollout(world, agent, starts, reward_fn,
                                    lambda o: np.ones(len(o), bool), {}, cfg_smoke)
assert diag['max_rollout_length'] <= 1, 'rollout continued past termination!'
print('termination respected inside model rollouts (max len under done-everywhere =',
      diag['max_rollout_length'], ')')

# (c) all four best checkpoints reload from Drive and evaluate
rows = trainer.evaluate_best(cfg_smoke, SMOKE_DIR, device='cuda', seeds=[0], eval_episodes=2)
assert len(rows) == 4, f'expected 4 reloadable checkpoints, got {len(rows)}'
display(pd.DataFrame(rows).round(2))
print('SMOKE GATE PASSED — pipeline sound. (Ordering at 1.5k steps is noise; it only means'
      ' something at 20k.)')

### 4.2 The full matrix — 4 algorithms × seeds → 20,000 real steps

The main event. With `SMOKE = False` this trains 12 runs (4 × 3 seeds) of 20k real environment steps
each; the ordering claim in Part 5 rests on these. The loop is **idempotent** — any run whose
`logs/<algo>_seed<seed>.json` already exists in Drive is reloaded, not retrained, so a Colab
disconnect costs only the run in flight (README §7: compute is the bottleneck, samples are not).
Each run prints one line per evaluation: greedy return on fixed shared eval seeds, failure rate, the
realised real/imagined mix, and a `<- best` marker whenever the best checkpoint is refreshed. With
`SMOKE = True` this cell simply reuses the smoke runs from 4.1.

In [ ]:
runs = []
for algo in cfg_run['experiment']['algorithms']:
    for seed in cfg_run['experiment']['seeds']:
        lp = f'{RUN_DIR}/logs/{algo}_seed{seed}.json'
        if os.path.exists(lp):
            print('skip (log exists):', algo, 'seed', seed)
            runs.append(json.load(open(lp)))
        else:
            runs.append(trainer.train_one(algo, cfg_run, RUN_DIR, seed=seed))
print('matrix complete:', len(runs), 'runs')

### 4.3 The UTD control run (fairness fix #5)

MACURA adapts its update-to-data ratio (Eq. 22) while MBPO/M2AC use a fixed G — so part of any
MACURA gain *could* be the update schedule rather than the adaptive rollouts. To rule that out we
retrain MACURA once (first seed) with `adaptive_gradient_steps: false`, i.e. G pinned to G_max like
the baselines, into `ablations/fixedG/`. Figure 5.9 overlays the two curves; if they land close,
rollout adaptation — not UTD — carries the win.

In [ ]:
ABL_DIR = f'{RUN_DIR}/ablations/fixedG'
os.makedirs(f'{ABL_DIR}/logs', exist_ok=True)
cfg_fixedG = copy.deepcopy(cfg_run)
cfg_fixedG['rollout']['macura']['adaptive_gradient_steps'] = False
seed0 = cfg_run['experiment']['seeds'][0]
lp = f'{ABL_DIR}/logs/macura_seed{seed0}.json'
if os.path.exists(lp):
    print('skip (log exists): macura fixed-G seed', seed0)
    run_fixedG = json.load(open(lp))
else:
    run_fixedG = trainer.train_one('macura', cfg_fixedG, ABL_DIR, seed=seed0)
run_adaptive = next(r for r in runs if r['algo'] == 'macura' and r['seed'] == seed0)
print('UTD control ready: adaptive vs fixed-G MACURA, seed', seed0)

## Part 5 — Results

Ten figures, each captioned with the claim it tests. Colors are fixed across every figure
(`viz/plots.ALGO_COLORS`): **MACURA blue, MBPO red, M2AC green, SAC gray**. Every figure is saved to
`{RUN_DIR}/plots/` *and* shown inline. Reminder of the pre-registered expectation: MACURA ≳ M2AC ≳
MBPO > SAC on sample efficiency; MACURA lowest failure rate and highest final return.

### 5.1 Sample efficiency — the headline figure

**Claim tested:** the model-based three learn in fewer real steps than SAC, and MACURA leads.
Mean ± std over seeds of greedy evaluation return vs real environment steps.

In [ ]:
plots.plot_sample_efficiency(runs, save_path=f'{PLOTS}/r51_sample_efficiency.png');
# Expected: blue (MACURA) on top and earliest to rise; red (MBPO) below/wobblier;
# gray (SAC) slowest — the model-free reference.

### 5.2 Per-seed spaghetti — honest variance

**Claim tested:** the 5.1 ordering is not an averaging artifact. Faint lines are individual seeds;
bold is the mean. RL variance is real — this figure is the honesty check on the headline.

In [ ]:
plots.plot_sample_efficiency_spaghetti(runs, save_path=f'{PLOTS}/r52_spaghetti.png');

### 5.3 Failure rate while learning — the stability story

**Claim tested:** MACURA crashes least on its way to competence (the safety/cost argument for
adaptive truncation: on a real drone every failure is bent hardware). Expected: MACURA lowest;
MBPO showing wobble/spikes once its scheduled horizon outruns model quality.

In [ ]:
plots.plot_failure_rate(runs, save_path=f'{PLOTS}/r53_failure_rate.png');

### 5.4 Final policy quality

**Claim tested:** MACURA doesn't just learn fast — it ends best. Best evaluation return per run,
mean ± std over seeds, per algorithm.

In [ ]:
plots.plot_final_quality(runs, save_path=f'{PLOTS}/r54_final_quality.png');

### 5.5 Steps to target — "how much faster, exactly?"

**Claim tested:** the sample-efficiency gap in one number: real steps until the evaluation return
first reaches a fixed target. We set the target at **80% of the best algorithm's mean best return**
(computed from the data, printed below) — high enough to mean "competent hover", low enough that
most algorithms reach it. A missing bar = never reached the target within the budget.

In [ ]:
by_algo = {}
for r in runs:
    by_algo.setdefault(r['algo'], []).append(max(r['eval_return']))
target = round(0.8 * max(np.mean(v) for v in by_algo.values()), 1)
print('target return =', target, '(80% of best algorithm mean best return)')
plots.plot_steps_to_target(runs, target_return=target,
                           save_path=f'{PLOTS}/r55_steps_to_target.png');

### 5.6 MACURA's signature: rollout length and κ over training

**Claim tested:** the adaptive mechanism actually adapts (project goal #3). Mean rollout length
should be short early (uncertain model, tight leash) and grow as the ensemble earns trust; κ
(Eq. 21) should fall as first-step uncertainties shrink. This figure is the direct, visible
demonstration of "trust the model where it trusts itself" — and thanks to fix #2, the GJS values
feeding it are scale-balanced rather than dominated by the position dims.

In [ ]:
plots.plot_rollout_depth(run_adaptive, save_path=f'{PLOTS}/r56_rollout_depth.png');

### 5.7 Model uncertainty over training

**Claim tested:** the ensemble genuinely learns the dynamics — the ζ-quantile of first-step GJS
uncertainty (the base of κ, Eq. 21) should fall over training. If this curve were flat or
oscillating wildly, κ would be meaningless and 5.6 an accident.

In [ ]:
plots.plot_gjs_over_training(run_adaptive, save_path=f'{PLOTS}/r57_gjs.png');

### 5.8 Rollout-length distribution, early vs late

**Claim tested:** adaptivity is per-rollout, not just an average drifting up. Early histogram mass
should sit at length 1–2; late mass should spread toward T_max — individual rollouts get exactly the
leash their local uncertainty allows.

In [ ]:
plots.plot_rollout_length_hist(run_adaptive, save_path=f'{PLOTS}/r58_len_hist.png');

### 5.9 The UTD control — adaptivity, not extra updates

**Claim tested:** MACURA's edge survives pinning G to G_max (fairness fix #5). If adaptive-G and
fixed-G MACURA land close (both above MBPO/M2AC), the rollout adaptation is what matters; if
fixed-G collapses toward the baselines, Eq. 22 deserves part of the credit and we say so.

In [ ]:
plots.plot_utd_ablation(run_adaptive, run_fixedG, save_path=f'{PLOTS}/r59_utd_control.png');

### 5.10 Audits: real/imagined mixing and wall-clock cost

Two bookkeeping figures. **Left/first:** the realised share of real-data batches in SAC updates
should track the fixed 5% `real_ratio` target (Janner/MBPO convention; identical for all three
model-based algorithms — if it drifted, mixing would be a confound). **Second:** return vs
wall-clock — the honest Colab caveat: MBRL wins on *samples* while costing more *compute* per step
than SAC (README §7).

In [ ]:
plots.plot_real_ratio(run_adaptive, save_path=f'{PLOTS}/r510_real_ratio.png')
plots.plot_return_vs_wallclock(runs, save_path=f'{PLOTS}/r510_wallclock.png');

## Part 5b — The flight demo: watch the result

Numbers convince reviewers; videos convince everyone else. For each algorithm we pick its **best
seed** (highest final greedy evaluation on the reloaded best checkpoint), load that checkpoint from
Drive, and record a **12-second deterministic flight clip** — same camera, same duration as the
random-policy "before" clip in 2.3. Clips save to `{RUN_DIR}/videos/<algo>_seed<seed>.mp4` via
`trainer.record_best_videos` → `viz.plots.record_policy_video`.

In [ ]:
best_seed = {}
for algo in cfg_run['experiment']['algorithms']:
    cand = [r for r in runs if r['algo'] == algo]
    best = max(cand, key=lambda r: (r.get('final_eval') or {}).get('eval_return', -np.inf))
    best_seed[algo] = best['seed']
print('best seed per algorithm:', best_seed)

video_paths = {}
for algo, seed in best_seed.items():
    cfg_v = copy.deepcopy(cfg_run)
    cfg_v['experiment']['algorithms'] = [algo]
    video_paths.update(trainer.record_best_videos(cfg_v, RUN_DIR, device='cuda',
                                                  seeds=[seed], seconds=12))
print('\nvideos on Drive:')
for name, path in video_paths.items():
    print(f'  {name}: {path}')

Each clip inline — the notebook itself is the demo. Watch for: MACURA holding a tight, level hover;
M2AC close behind; MBPO drifting or oscillating; SAC (with only 20k real steps) visibly the least
composed.

In [ ]:
for name, path in video_paths.items():
    print(name)
    display(Video(path, embed=True, width=440))

And the one-glance version: all four policies tiled into a single side-by-side reel
(`viz.plots.stitch_videos_grid` — labels burned in, shorter clips hold their last frame).

In [ ]:
labels = {name.split('_seed')[0]: path for name, path in video_paths.items()}
reel = f'{RUN_DIR}/videos/side_by_side_all4.mp4'
plots.stitch_videos_grid(labels, reel, fps=30, cols=2, downscale=2)
print('saved', reel)
display(Video(reel, embed=True, width=640))

Finally, confirm everything actually landed in Drive, and optionally pull the files to your machine.

In [ ]:
vids = sorted(glob.glob(f'{RUN_DIR}/videos/*.mp4'))
print(f'{len(vids)} video(s) in {RUN_DIR}/videos:')
for v in vids:
    print(f'  {os.path.basename(v):40s} {os.path.getsize(v)/1e6:6.1f} MB')
assert any('seed' in v for v in vids), 'no per-algorithm videos were written!'

DOWNLOAD_VIDEOS = False    # set True to download every mp4 through the browser
if DOWNLOAD_VIDEOS:
    from google.colab import files
    for v in vids:
        files.download(v)

## Part 6 — Discussion

### 6.1 Reading the results against the Part-3 hypothesis

Three questions were on trial (`project.md` §6), and the honest answers come from the figures, not
from this paragraph: **Did the model help?** (MACURA vs SAC — figures 5.1/5.5: the model-based gap
in real steps.) **Did adaptivity matter?** (MACURA vs MBPO — 5.1/5.3: whether the tuned fixed
schedule wobbles where the adaptive leash stays stable, and 5.6 showing the leash actually
adapting.) **Truncation vs masking?** (MACURA vs M2AC — 5.1/5.4: two uses of the *same* GJS signal,
head-to-head.) The cell below computes the measured ordering from the final evaluations and states
plainly whether the pre-registered hypothesis held — including which link failed, if one did. If
this is a smoke run it says so and refuses to over-read 1.5k-step noise.

In [ ]:
summary = []
for algo in cfg_run['experiment']['algorithms']:
    rl = [r for r in runs if r['algo'] == algo]
    fe = [(r.get('final_eval') or {}) for r in rl]
    summary.append({
        'algo': algo,
        'final_return_mean': np.mean([f.get('eval_return', np.nan) for f in fe]),
        'final_return_std':  np.std([f.get('eval_return', np.nan) for f in fe]),
        'final_failure_rate': np.mean([f.get('eval_failure_rate', np.nan) for f in fe]),
        'best_return_mean':  np.mean([r.get('best_return', np.nan) for r in rl]),
    })
df = pd.DataFrame(summary).set_index('algo')
display(df.round(2))

order = df['final_return_mean'].sort_values(ascending=False)
print('measured ordering (final return):', ' > '.join(a.upper() for a in order.index))
most_stable = df['final_failure_rate'].idxmin()
print('lowest final failure rate:', most_stable.upper())

g = lambda a: float(df.loc[a, 'final_return_mean'])
links = {
    'MACURA >= M2AC (truncation beats masking)':      g('macura') >= g('m2ac'),
    'M2AC >= MBPO (uncertainty use beats schedule)':  g('m2ac') >= g('mbpo'),
    'MACURA > SAC (the model helps)':                 g('macura') > g('sac'),
    'MACURA most stable (lowest failure rate)':       most_stable == 'macura',
}
if SMOKE:
    print('\nNOTE: SMOKE run — 1.5k-step ordering is noise. Set SMOKE = False and rerun'
          ' Parts 4-6 for the verdict that counts.')
elif all(links.values()):
    print('\nVERDICT: the hypothesis HOLDS on this 20k-step run — MACURA leads on final'
          ' return and stability, M2AC ≥ MBPO, all model-based above SAC. See 5.1/5.3/5.4.')
else:
    print('\nVERDICT: the hypothesis did NOT fully hold. Failed link(s):')
    for name, ok in links.items():
        if not ok:
            print('  ✗', name)
    print('Likely causes to check before re-running (do NOT tune on the result):'
          '\n  - kappa/GJS pathology: is 5.7 falling and 5.6 growing? flat GJS => ensemble'
          '\n    underfit (raise ensemble.train_epochs_per_round) or xi mis-set (Phase 5 sweep);'
          '\n  - seed variance: 3 seeds is thin — raise experiment.seeds to 5 (tasks.md Phase 6);'
          '\n  - MBPO schedule: if MBPO ~ MACURA, the drone may tolerate length-10 rollouts —'
          '\n    the honest conclusion would be that adaptivity buys little here;'
          '\n  - budget: if nothing beats SAC clearly, 20k steps may be short of takeoff —'
          '\n    inspect 5.1 slopes near 20k before extending.')

### 6.2 Fairness audit — did the correctness-gate fixes hold?

Re-assert, from the *resolved config and the actual run logs*, every fairness guarantee from Part 3.
This cell failing loudly is a feature: a broken guarantee means the Part-5 conclusions are void.

In [ ]:
c = cfg_run
gjs_vals = [v for _, v in run_adaptive.get('gjs', [])]
sched = c['rollout']['mbpo']['rollout_schedule']
checks = [
    ('#1 one exploration scheme (pink) for all four algorithms',
     c['exploration']['type'] == 'pink_noise'),
    ('#2 GJS finite & positive across training (scale-balanced targets)',
     len(gjs_vals) > 0 and all(np.isfinite(gjs_vals)) and min(gjs_vals) > 0),
    ('#3 SAC baseline UTD = 1 (standard, not strawman)',
     int(c['sac']['baseline_gradient_steps']) == 1),
    ('#4 MBPO schedule ramps within the budget',
     sched[3] <= cfg_full['experiment']['total_env_steps']),
    ('#5 fixed-G UTD control exists and matches Gmax parity',
     os.path.exists(f'{ABL_DIR}/logs/macura_seed{seed0}.json')
     and c['rollout']['mbpo']['fixed_gradient_steps'] == c['sac']['gradient_steps_max']
     and c['rollout']['m2ac']['fixed_gradient_steps'] == c['sac']['gradient_steps_max']),
    ('#6 SB3 API verified on this runtime (cell 4.1 ran)',
     True),  # cell 4.1 asserts hard; reaching here means it passed
    ('shared eval seeds, unchanged from the config (fairness)',
     c['experiment']['eval_seeds'] == CFG['experiment']['eval_seeds']
     and len(c['experiment']['eval_seeds']) >= 5),
]
for name, ok in checks:
    print('PASS ' if ok else 'FAIL ', name)
assert all(ok for _, ok in checks), 'fairness audit failed — Part 5 conclusions are void'
print('\nfairness audit PASSED')

### 6.3 Reproducibility footer

Everything needed to reproduce or audit this run: seeds, the SHA-256 of the resolved config, library
versions, and the Drive path of every artifact this notebook produced.

In [ ]:
resolved = {k: v for k, v in cfg_run.items() if k != 'profiles'}
cfg_hash = hashlib.sha256(json.dumps(resolved, sort_keys=True).encode()).hexdigest()[:16]
print('config sha256(16):', cfg_hash, '| SMOKE =', SMOKE)
print('train seeds      :', cfg_run['experiment']['seeds'],
      '| eval seeds:', cfg_run['experiment']['eval_seeds'])
print('versions         : torch', torch.__version__, '| sb3', stable_baselines3.__version__,
      '| pybullet', PYB_VER, '| gymnasium', gymnasium.__version__, '| numpy', np.__version__)
print('GPU              :', torch.cuda.get_device_name(0))
for sub in ('logs', 'checkpoints', 'plots', 'videos', 'ablations/fixedG/logs'):
    fs = sorted(glob.glob(f'{RUN_DIR}/{sub}/*'))
    print(f'\n{RUN_DIR}/{sub}  ({len(fs)} files)')
    for f in fs:
        print('   ', os.path.basename(f))

### What to run next (tied to `tasks.md`)

- **5 seeds for the final figures** — `experiment.seeds` → `[0,1,2,3,4]` (Phase 6); 3 seeds is the
  minimum for legible bands, not the last word.
- **ξ sweep** — `rollout.macura.xi` over {1, 3, 5, 10, 20} as in the paper (Phase 5); ξ is MACURA's
  one knob and 5.6/5.7 tell you whether the current 5.0 leash is too tight or too loose.
- **MuJoCo/Hopper anchor** — run this identical code on a stock locomotion task (Phase 6) to
  confirm it reproduces the paper qualitatively, de-risking "drone result" vs "bug".
- **Faithful M2AC variant** — per-step rank-based masking with the OvR estimate + penalty
  (Phase 4, med) to match Pan et al. exactly rather than our GJS-based, M2AC-favouring version.
- **Robustness & extensions** — sensor/action noise via `NoisyObsWrapper` (does GJS still isolate
  epistemic uncertainty?), the `track` task, and the MuJoCo Skydio X2 backend A/B (Phase 7).